In [1]:
transactions = [
    {"Roti", "Selai", "Mentega"},
    {"Roti", "Mentega"},
    {"Roti", "Susu", "Mentega"},
    {"Cokelat", "Roti", "Susu", "Mentega"},
    {"Cokelat", "Susu"}
]

In [2]:
min_support = 0.3  # 30%
min_confidence = 0.6  # 60%

In [3]:
from itertools import combinations
from collections import defaultdict

In [4]:
def calculate_support(transactions, itemset):
    count = sum(1 for transaction in transactions if itemset.issubset(transaction))
    return count / len(transactions)

In [5]:
unique_items = set().union(*transactions)

In [6]:
support_data = {}
for item in unique_items:
    itemset = frozenset([item])
    support = calculate_support(transactions, itemset)
    if support >= min_support:
        support_data[itemset] = support

In [7]:
k = 2
frequent_itemsets = list(support_data.keys())
while frequent_itemsets:
    candidates = [frozenset(a | b) for a in frequent_itemsets for b in frequent_itemsets if len(a | b) == k]
    candidates = set(candidates)  # Remove duplicates
    valid_candidates = {}
    for candidate in candidates:
        support = calculate_support(transactions, candidate)
        if support >= min_support:
            valid_candidates[candidate] = support
    if not valid_candidates:
        break
    support_data.update(valid_candidates)
    frequent_itemsets = list(valid_candidates.keys())
    k += 1

In [8]:
rules = []
for itemset in support_data.keys():
    if len(itemset) > 1:
        for antecedent_size in range(1, len(itemset)):
            for antecedent in combinations(itemset, antecedent_size):
                antecedent = frozenset(antecedent)
                consequent = itemset - antecedent
                confidence = support_data[itemset] / support_data[antecedent]
                if confidence >= min_confidence:
                    rules.append((antecedent, consequent, confidence))

support_data, rules

({frozenset({'Roti'}): 0.8,
  frozenset({'Mentega'}): 0.8,
  frozenset({'Susu'}): 0.6,
  frozenset({'Cokelat'}): 0.4,
  frozenset({'Roti', 'Susu'}): 0.4,
  frozenset({'Cokelat', 'Susu'}): 0.4,
  frozenset({'Mentega', 'Roti'}): 0.8,
  frozenset({'Mentega', 'Susu'}): 0.4,
  frozenset({'Mentega', 'Roti', 'Susu'}): 0.4},
 [(frozenset({'Susu'}), frozenset({'Roti'}), 0.6666666666666667),
  (frozenset({'Susu'}), frozenset({'Cokelat'}), 0.6666666666666667),
  (frozenset({'Cokelat'}), frozenset({'Susu'}), 1.0),
  (frozenset({'Roti'}), frozenset({'Mentega'}), 1.0),
  (frozenset({'Mentega'}), frozenset({'Roti'}), 1.0),
  (frozenset({'Susu'}), frozenset({'Mentega'}), 0.6666666666666667),
  (frozenset({'Susu'}), frozenset({'Mentega', 'Roti'}), 0.6666666666666667),
  (frozenset({'Roti', 'Susu'}), frozenset({'Mentega'}), 1.0),
  (frozenset({'Mentega', 'Susu'}), frozenset({'Roti'}), 1.0)])